<a href="https://colab.research.google.com/github/karsarobert/LLM2026/blob/main/ChatGPT_12_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Chat GPT és más nagy nyelvi modellek alkalmazása
##PTE Gépi tanulás
###12. gyakorlat: Langchain 2. rész
2026. április 27.

# LangChain oktatóanyag: LLM alkalmazások fejlesztése

Ez az anyag a LangChain keretrendszer használatát mutatja be nagy nyelvi modelleken (LLM-eken) alapuló alkalmazások fejlesztéséhez. A notebook az **aktuálisabb, LangChain v1-hez jobban illeszkedő** megközelítéseket követi: külön integrációs csomagokat, `create_agent` alapú ügynököket, modern promptkezelést és korszerűbb memóriakezelést használ.

Ebben a változatban a chat modellekhez **Groq OpenAI-kompatibilis végpontot** használunk `ChatOpenAI(base_url="https://api.groq.com/openai/v1", ...)` mintával. A retrieval részben az embeddingeket nem OpenAI API-val, hanem **helyi Hugging Face embedding modellel** állítjuk elő.

A notebook fő témái:
- ügynökök és eszközhasználat,
- nyílt és zárt modellek használata,
- beszélgetési memória,
- RAG és agentic RAG,
- modern LangChain runnable szemlélet.


In [ ]:
# Szükséges csomagok telepítése
!pip install -U langchain langchain-openai langchain-community langchain-huggingface langchain-text-splitters openai google-search-results pypdf faiss-cpu sentence-transformers tiktoken -q


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts

**Magyarázat:**

Ebben a notebookban a LangChain újabb, moduláris felépítését használjuk, ezért több külön csomagot telepítünk:

- `langchain`: az alap LangChain keretrendszer
- `langchain-openai`: OpenAI-kompatibilis chat modellekhez; ezt most a **Groq kompatibilis végpontjával** használjuk
- `langchain-community`: közösségi integrációkhoz, pl. dokumentumbetöltők és vektortárolók
- `langchain-huggingface`: Hugging Face modellekhez és embeddingekhez
- `langchain-text-splitters`: szövegdarabolókhoz
- `openai`: az OpenAI-kompatibilis kliensréteghez, amelyet a `ChatOpenAI` is használ
- `google-search-results`: SerpAPI webes kereséshez https://serpapi.com/
- `pypdf`: PDF-ek feldolgozásához
- `faiss-cpu`: vektoralapú hasonlósági kereséshez
- `sentence-transformers`: helyi embedding modellekhez
- `tiktoken`: tokenizálási feladatokhoz

A chat-modell részeknél tehát **nem OpenAI API-kulcsot**, hanem **Groq API-kulcsot** használunk.


In [ ]:
# API-kulcsok beolvasása (Colab Secrets használata ajánlott)
from google.colab import userdata
import os

# Helyettesítsd be a Secret neveket, amiket Colabban megadtál
# os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
# os.environ['SERPAPI_API_KEY'] = userdata.get('SERPAPI_API_KEY')
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

GROQ_API_KEY = userdata.get("GROQ_API_KEY")
SERPAPI_API_KEY = userdata.get("SERPAPI_API_KEY")
HF_TOKEN = userdata.get("HF_TOKEN")  # csak akkor kell, ha a HF példához szükséges

print("GROQ kulcs betöltve:", bool(GROQ_API_KEY))
print("SERPAPI kulcs betöltve:", bool(SERPAPI_API_KEY))
print("HF token betöltve:", bool(HF_TOKEN))


GROQ kulcs betöltve: True
SERPAPI kulcs betöltve: True
HF token betöltve: True


---

## 2. Ügynökök (Agents)

Az agent olyan rendszer, amely egy modellt eszközökkel (tools) köt össze. A modell eldöntheti, hogy elég-e a saját tudása a válaszadáshoz, vagy érdemes valamilyen külső eszközt meghívnia.

A modern LangChain-ben új fejlesztésekhez a `create_agent(...)` a javasolt belépési pont.

**Példa:**


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_community.utilities import SerpAPIWrapper
from google.colab import userdata

# 1. Kereső inicializálása
search = SerpAPIWrapper(serpapi_api_key=userdata.get("SERPAPI_API_KEY"))

@tool
def web_search(query: str) -> str:
    """Hasznos, ha aktuális vagy naprakész információkra van szükség."""
    return search.run(query)

# 2. Groq alapú chat modell inicializálása OpenAI-kompatibilis végponttal
llm = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEY"),
    model="openai/gpt-oss-120b",
    temperature=0
)

# 3. Agent létrehozása
agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=(
        "Te egy segítőkész kutatóasszisztens vagy. "
        "Ha a kérdés naprakész vagy webes információt igényel, használd a web_search eszközt."
    )
)

# 4. Futtatás
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Ki volt Arany János felesége és mikor született?"}
        ]
    }
)

print("\nVégső válasz:")
print(response["messages"][-1].content)



Végső válasz:
Arany János felesége **Ercsey Julianna** volt, akit gyakran Arany Juliannak vagy egyszerűen „Juliskának” is emlegetnek.  

- **Születési dátuma:** 1818. május 1. (Nagyszalonta)【3†L1-L3】  

Ercsey Julianna 1818. május 1‑jén született Nagyszalontán, és 1840. november 19‑én házasodott össze Arany Jánossal. A házasságukból két gyermek született: Arany Juliska (1841. augusztus 9.) és Arany László (1844. március 24.).  

Tehát Arany János felesége Ercsey Julianna, születési dátuma 1818. május 1.


#HTML megjelenítő

In [ ]:
from IPython.display import display, HTML
from html import escape
import json

def render_agent_timeline(response):
    messages = response["messages"]

    css = """
    <style>
      .agent-wrap {font-family: Inter, Arial, sans-serif; line-height: 1.45;}
      .agent-card {
        border: 1px solid #e5e7eb;
        border-left: 6px solid #d1d5db;
        border-radius: 14px;
        padding: 14px 16px;
        margin: 12px 0;
        box-shadow: 0 2px 10px rgba(0,0,0,0.04);
      }
      .human {background:#eff6ff; border-left-color:#3b82f6;}
      .ai {background:#f9fafb; border-left-color:#6b7280;}
      .toolcall {background:#fff7ed; border-left-color:#f59e0b;}
      .tool {background:#ecfdf5; border-left-color:#10b981;}
      .final {background:#f0fdf4; border-left-color:#22c55e;}
      .meta {font-size: 12px; color:#6b7280; margin-bottom: 8px;}
      .title {font-weight: 700; margin-bottom: 6px;}
      .content {white-space: pre-wrap;}
      details {margin-top: 8px;}
      summary {cursor: pointer; font-weight: 600;}
      code, pre {
        background:#111827; color:#f9fafb;
        border-radius: 10px; padding: 10px;
        display:block; overflow-x:auto;
      }
      .small {font-size: 12px; color:#4b5563; margin-top:8px;}
    </style>
    """

    blocks = ['<div class="agent-wrap">', css]

    for i, msg in enumerate(messages, start=1):
        cls_name = type(msg).__name__

        # 1) HumanMessage
        if cls_name == "HumanMessage":
            blocks.append(f"""
            <div class="agent-card human">
              <div class="title">🧑 Felhasználó</div>
              <div class="content">{escape(str(msg.content))}</div>
            </div>
            """)

        # 2) AIMessage - lehet tool call vagy végső válasz
        elif cls_name == "AIMessage":
            tool_calls = getattr(msg, "tool_calls", []) or []
            content = getattr(msg, "content", "") or ""
            meta = getattr(msg, "response_metadata", {}) or {}
            usage = meta.get("token_usage", {})
            finish_reason = meta.get("finish_reason", "")

            if tool_calls:
                tool_html = []
                for tc in tool_calls:
                    name = tc.get("name", "tool")
                    args = tc.get("args", {})
                    tool_html.append(f"""
                    <div style="margin:8px 0; padding:10px; background:white; border-radius:10px; border:1px solid #fed7aa;">
                      <b>Tool:</b> {escape(name)}<br>
                      <b>Args:</b>
                      <pre>{escape(json.dumps(args, ensure_ascii=False, indent=2))}</pre>
                    </div>
                    """)

                blocks.append(f"""
                <div class="agent-card toolcall">
                  <div class="title">🤖 Modell döntés</div>
                  <div class="meta">finish_reason = {escape(str(finish_reason or "tool_calls"))}</div>
                  <div class="content">A modell eszközhívást kezdeményezett.</div>
                  {''.join(tool_html)}
                  <div class="small">
                    prompt_tokens={usage.get("prompt_tokens", "-")},
                    completion_tokens={usage.get("completion_tokens", "-")},
                    total_tokens={usage.get("total_tokens", "-")}
                  </div>
                </div>
                """)

            elif content.strip():
                blocks.append(f"""
                <div class="agent-card final">
                  <div class="title">✅ Végső válasz</div>
                  <div class="content">{escape(content)}</div>
                  <div class="small">
                    prompt_tokens={usage.get("prompt_tokens", "-")},
                    completion_tokens={usage.get("completion_tokens", "-")},
                    total_tokens={usage.get("total_tokens", "-")}
                  </div>
                </div>
                """)

            else:
                blocks.append(f"""
                <div class="agent-card ai">
                  <div class="title">🤖 AIMessage</div>
                  <div class="content"><i>Üres tartalom</i></div>
                </div>
                """)

        # 3) ToolMessage
        elif cls_name == "ToolMessage":
            name = getattr(msg, "name", "tool")
            content = getattr(msg, "content", "")
            blocks.append(f"""
            <div class="agent-card tool">
              <div class="title">🛠️ Tool eredmény: {escape(str(name))}</div>
              <details>
                <summary>Tool output megnyitása</summary>
                <pre>{escape(str(content))}</pre>
              </details>
            </div>
            """)

        # 4) bármi más
        else:
            blocks.append(f"""
            <div class="agent-card ai">
              <div class="title">📦 {escape(cls_name)}</div>
              <pre>{escape(str(msg))}</pre>
            </div>
            """)

    blocks.append("</div>")
    display(HTML("".join(blocks)))

#Agent működése

In [ ]:
render_agent_timeline(response)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from google.colab import userdata

# 1. Groq modell
llm_chat = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEY"),
    model="openai/gpt-oss-120b",
    temperature=0.2
)

# 2. Prompt
prompt_chat = ChatPromptTemplate.from_messages([
    ("system", "Ön egy barátságos és segítőkész asszisztens."),
    MessagesPlaceholder("history"),
    ("human", "{input}")
])

base_chain = prompt_chat | llm_chat

# 3. Egyszerű in-memory history store
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 4. Lánc memóriával
chain_with_memory = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

session_id = "vienna-demo"

def chat_with_memory_func(user_input: str):
    response = chain_with_memory.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}},
    )
    return response.content



In [ ]:
# Használat
print("AI:", chat_with_memory_func("Szia! Segítenél nekem?"))



AI: Szia! Persze, szívesen segítek. Miben lehetek a segítségedre?


In [ ]:
print("AI:", chat_with_memory_func("Szeretnék Bécsbe utazni 2 napra. Mit javasolsz megnézni?"))


AI: **Bécs 2‑napos programjavaslat**  
*(az úticélok közel vannak egymáshoz, így könnyen gyalog vagy tömegközlekedéssel elérhetők)*  

---

## 1. nap – Klasszikus Bécs & a belváros

| Idő | Program | Rövid leírás & tipp |
|-----|---------|---------------------|
| **09:00** | **Reggeli a Café Central** | Legendás kávéház, ahol Sacher‑torta, kávé és friss croissant vár. Foglalj asztalt előre, ha hétvégén mész. |
| **10:00** | **Stephansdom (Szent István-székesegyház)** | A város szíve. Lépj fel a toronyba (kb. 8 €/fő) a kilátásért a Duna-öbölre. |
| **11:30** | **Graben & Kohlmarkt** | Sétálj a luxus vásárlóutcákon, nézd meg a **Pestsäule** (csodálatos oszlop) és a **Kohlmarkt** elegáns butikjait. |
| **12:30** | **Ebéd a Naschmarkt** | Piac tele nemzetközi standokkal. Próbáld ki a **Wiener Schnitzel** vagy a **Kebap** a magyar standnál. |
| **14:00** | **Hofburg Palota** | A Habsburgok egykori rezidenciája. Látogasd meg a **Sisi Múzeumot**, a **Kincstárat** és a **Schatzkammer**-t (kb. 

In [ ]:
print("AI:", chat_with_memory_func("Említetted a Prátert. Mesélnél róla bővebben?"))



AI: ## A Práter – Bécs legnagyobb, legszínesebb szabadidő‑ és szórakoztatóparkja  

| Téma | Részletek |
|------|-----------|
| **Helyszín** | A Donau‑sziget (Donauinsel) és a Bécsi‑Duna közötti terület, a 2. kerület (Leopoldstadt) déli részén. A park mintegy **6 km²** területet foglal el, így könnyen egy egész napot is eltölthetsz itt. |
| **Történelem** | - **1766**: a területet a Habsburgok vadászterületnek („Jagd‑ und Lust‑Garten”) nevezték. <br>- **1873**: a **Wiener Weltausstellung** (világtárgyalás) keretében nyílt meg a **„Wiener Prater”** – ekkor indult el a mai értelemben vett szórakoztatópark. <br>- **1901**: felállították a híres **Riesenrad‑t (óriáskereket)**, amely azóta Bécs egyik szimbóluma. |
| **Főbb attrakciók** | 1. **Riesenrad (óriáskerék)** – 65 m magasságú, 24 kabinnal. 8‑10 perc alatt egy teljes kör. <br>2. **Prater‑Allee** – 4,5 km hosszú, fák sorával szegélyezett sétány, kerékpározásra, futásra, görkorcsolyázásra is alkalmas. <br>3. **Művészeti és szórakoztató

In [ ]:
print("\nMemória tartalma:")
print(store[session_id].messages)


Memória tartalma:
[HumanMessage(content='Szia! Segítenél nekem?', additional_kwargs={}, response_metadata={}), AIMessage(content='Szia! Persze, szívesen segítek. Miben lehetek a segítségedre?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 102, 'total_tokens': 182, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 46, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'queue_time': 0.004977244, 'prompt_time': 0.003773417, 'completion_time': 0.168550999, 'total_time': 0.172324416}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_1d982b31b2', 'id': 'chatcmpl-f20a1998-34e5-442d-b1ad-bb688226c87f', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dce9e-5bc5-79d0-9c91-e573dd5988de-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 102, 'output_to

**Magyarázat:**

- A `RunnableWithMessageHistory` egy meglévő runnable köré épít beszélgetési előzményt.
- Az `InMemoryChatMessageHistory` egy egyszerű, memóriában tárolt üzenettörténet. Oktatási példákhoz jó, de éles rendszerben általában tartós tárolást használunk.
- A `MessagesPlaceholder("history")` helyére kerülnek be a korábbi üzenetek.
- A `session_id` választja szét a külön beszélgetésszálakat.
- A modell itt is **Groq-on fut**, csak a LangChain oldalon továbbra is `ChatOpenAI` interfészt használunk az OpenAI-kompatibilitás miatt.
- Ez a megközelítés jobban illeszkedik a mai LangChain szemlélethez, mint a régebbi `ConversationBufferMemory`-alapú minta.

---

## 5. RAG memória és retrieval segítségével

Most építsünk egy olyan láncot, amely PDF-ből származó információt használ fel a válaszadáshoz.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import get_buffer_string
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from operator import itemgetter
from google.colab import userdata
import requests
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

# --- Setup ---
llm_rag = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEY"),
    model="openai/gpt-oss-120b",
    temperature=0
)

chat_history_rag = InMemoryChatMessageHistory()

# A PyPDFLoader tipikusan helyi fájlra dolgozik, ezért a PDF-et előbb letöltjük.
pdf_url = "https://www.wien.info/resource/blob/412434/49dcd5d77c516144b39195951e771b83/route-wege-die-stadt-kennen-zu-lernen-en-data.pdf"
local_pdf_path = "/content/vienna_guide.pdf"

try:
    print(f"PDF letöltése innen: {pdf_url}")
    response = requests.get(pdf_url, timeout=30)
    response.raise_for_status()
    with open(local_pdf_path, "wb") as f:
        f.write(response.content)

    loader = PyPDFLoader(local_pdf_path)
    raw_documents = loader.load()
    print(f"Dokumentum betöltve, {len(raw_documents)} oldal.")
except Exception as e:
    print(f"Hiba a PDF betöltése közben: {e}")
    raw_documents = []

if raw_documents:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150
    )
    documents = text_splitter.split_documents(raw_documents)
    print(f"Dokumentum felosztva {len(documents)} darabra.")

    # A chat modell Groq-on fut, az embeddingeket viszont helyi Hugging Face modellel készítjük.
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    print("Vektoradatbázis létrehozása...")
    vectorstore = FAISS.from_documents(documents, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    print("Retriever kész.")
else:
    retriever = None
    print("Nincsenek dokumentumok a RAG lánc építéséhez.")

# --- Prompts ---
condense_question_prompt = ChatPromptTemplate.from_messages([
    ("system", "Fogalmazd át a felhasználó kérdését önálló keresési lekérdezéssé a beszélgetési előzmény alapján."),
    ("human", "Beszélgetési előzmény:\n{chat_history}\n\nFelhasználói kérdés: {input}")
])

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Ön egy segítőkész asszisztens. Csak a megadott kontextus alapján válaszoljon. Ha a kontextus nem tartalmazza a választ, mondja ezt világosan."),
    ("human", "Kérdés: {input}\n\nÖnálló kérdés: {standalone_question}\n\nKontextus:\n{context}")
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def get_chat_history_string(_):
    return get_buffer_string(chat_history_rag.messages)

if retriever:
    rag_chain = (
        RunnablePassthrough.assign(
            chat_history=RunnableLambda(get_chat_history_string)
        )
        .assign(
            standalone_question=condense_question_prompt | llm_rag | StrOutputParser()
        )
        .assign(
            context=itemgetter("standalone_question") | retriever | RunnableLambda(format_docs)
        )
        .assign(
            answer=answer_prompt | llm_rag | StrOutputParser()
        )
    )
    print("RAG lánc sikeresen létrehozva.")
else:
    rag_chain = None
    print("RAG lánc nem jött létre.")

def chat_rag_with_memory(user_input: str):
    if not rag_chain:
        return "Sajnos a RAG lánc nem érhető el a dokumentumok betöltési hibája miatt."

    result = rag_chain.invoke({"input": user_input})

    chat_history_rag.add_user_message(user_input)
    chat_history_rag.add_ai_message(result["answer"])

    return result["answer"]

# --- Teszt ---
if rag_chain:
    print("\n--- RAG teszt ---")
    print("Felhasználó: Mi a Mozart-ház érdekessége?")
    print("AI:", chat_rag_with_memory("Mi a Mozart-ház érdekessége?"))
else:
    print("\nA RAG lánc nem tesztelhető a korábbi hiba miatt.")


PDF letöltése innen: https://www.wien.info/resource/blob/412434/49dcd5d77c516144b39195951e771b83/route-wege-die-stadt-kennen-zu-lernen-en-data.pdf
Dokumentum betöltve, 20 oldal.
Dokumentum felosztva 60 darabra.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vektoradatbázis létrehozása...
Retriever kész.
RAG lánc sikeresen létrehozva.

--- RAG teszt ---
Felhasználó: Mi a Mozart-ház érdekessége?
AI: A megadott szöveg szerint a **Mozarthaus Vienna** (Mozart-ház) az a hely, ahol Wolfgang Amadeus Mozart 1784‑től 1787‑ig élt. Érdekessége, hogy ebben az eredeti lakásban tartotta a híres, számtalan vendégnek szóló parti‑rendezvényeit. Ez teszi különlegessé a házat a látogatók számára.


**Magyarázat:**

- A dokumentumot először betöltjük, majd kisebb chunkokra bontjuk.
- A chat modell a **Groq OpenAI-kompatibilis végpontját** használja.
- Az embeddingekhez itt **nem OpenAI API-t** használunk, hanem egy helyi Hugging Face modellt (`sentence-transformers/all-MiniLM-L6-v2`).
- A FAISS vektortároló gyors hasonlósági keresést tesz lehetővé.
- A lánc először egy önálló keresési lekérdezést készít a felhasználói kérdésből és az előzményekből.
- Ezután a retriever megkeresi a releváns dokumentumrészeket.
- Végül a modell kizárólag a visszakeresett kontextus alapján válaszol.

Ez a példa nem klasszikus `RetrievalQA`-t használ, hanem egy modernebb, runnable alapú összeállítást.


In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SerpAPIWrapper
from google.colab import userdata

llm_agent_rag = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("GROQ_API_KEY"),
    model="openai/gpt-oss-120b",
    temperature=0
)

# Retriever alapú tool
if retriever:
    @tool
    def austria_guide_search(query: str) -> str:
        """Használd, ha Ausztriával, Béccsel vagy a bécsi útikalauz tartalmával kapcsolatos kérdésre kell válaszolni."""
        docs = retriever.invoke(query)
        return "\n\n".join(doc.page_content for doc in docs)
else:
    austria_guide_search = None

# Webes kereső tool
search_agent = SerpAPIWrapper(serpapi_api_key=userdata.get("SERPAPI_API_KEY"))

@tool
def general_search(query: str) -> str:
    """Használd, ha naprakész vagy általános webes információra van szükség."""
    return search_agent.run(query)

tools_agent = [general_search]
if austria_guide_search:
    tools_agent.append(austria_guide_search)

agent_rag_tool = create_agent(
    model=llm_agent_rag,
    tools=tools_agent,
    system_prompt=(
        "Te egy segítőkész asszisztens vagy. "
        "A bécsi útikalauzzal kapcsolatos kérdéseknél használd az austria_guide_search eszközt. "
        "Általános vagy naprakész kérdéseknél használd a general_search eszközt."
    )
)

# --- Teszt ---
print("\n--- Agent teszt (retriever + webes keresés) ---")

response1 = agent_rag_tool.invoke(
    {"messages": [{"role": "user", "content": "Mesélj nekem a Mozartházról a bécsi útikalauz alapján!"}]}
)
print("AI:", response1["messages"][-1].content)

response2 = agent_rag_tool.invoke(
    {"messages": [{"role": "user", "content": "Milyen az időjárás most Budapesten?"}]}
)
print("AI:", response2["messages"][-1].content)



--- Agent teszt (retriever + webes keresés) ---
AI: **Mozartház – Bécsi útikalauz szerint**

A **Mozartház (Mozarthaus Vienna)** a bécsi útikalauz egyik kiemelt látnivalója, amely a klasszikus zene szerelmeseinek kötelező állomása. Az alábbiakban összegyűjtöttük a legfontosabb információkat, amelyeket a bécsi útikalauz (a *Vienna Guide*) tartalmaz.

---

### 1. Alapvető adatok
| **Információ** | **Részletek** |
|----------------|---------------|
| **Cím** | Domgasse 5, 1010 Bécs |
| **Weboldal** | [mozarthausvienna.at](https://mozarthausvienna.at) |
| **Nyitvatartás** | Hétfőtől vasárnapig 10:00‑18:00 (utolsó belépés 17:30). Zárva tart a 1. január és a 25. december. |
| **Belépő** | Felnőttek: €12, diákok/nyugdíjasok: €9, családcsomag (2 felnőtt + 2 gyermek): €30. A *Vienna Pass* kártyával ingyenes belépés. |

---

### 2. Történelmi háttér
- **Eredeti lakás**: A Mozartház a **Domgasse 5‑ben** található épület eredeti lakása, ahol Wolfgang Amadeus Mozart 1784‑1787 között élt. Ebben az 

In [ ]:
render_agent_timeline(response1)

In [ ]:
render_agent_timeline(response2)

**Magyarázat:**

- Az agent egyszerre több eszközt is kaphat.
- A `general_search` webes, naprakész információkhoz használható.
- Az `austria_guide_search` a belső dokumentumalapú tudásbázisban keres.
- A modell itt is **Groq-on fut**, a toolokat pedig a LangChain agent rétegen keresztül használja.
- Az agent a kérdés jellege alapján választja ki a megfelelő eszközt.
- Ez a megközelítés jól szemlélteti az **agentic RAG** egyik alapformáját: a modell nem mindig ugyanúgy jár el, hanem dinamikusan dönt a retrieval szükségességéről.

---

## 7. Beadandó feladat

Készíts egy olyan LangChain-alapú alkalmazást, amelyben legalább:
- van egy agent vagy runnable alapú folyamat,
- szerepel legalább két tool,
- és megjelenik valamilyen retrieval vagy dokumentumalapú feldolgozás.


**Határidő:** 2026.05.20

Javaslat a beadandó dokumentálásához:
- röviden írd le az architektúrát,
- mutasd meg az agenthez adott eszközöket,
- adj 2–3 demonstrációs lekérdezést,
- jelezd, melyik kérdésnél melyik eszköz volt releváns,
- és írd le a korlátokat vagy tapasztalt hibákat is.
